In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))
from config.paths import RAW_DATA_PATH

In [4]:
import pandas as pd
import numpy as np

from config.paths import RAW_DATA_PATH
from src.pipeline import run_pipeline

df_raw = pd.read_csv(RAW_DATA_PATH, parse_dates=["date"])
result = run_pipeline(df_raw)

partitions = {
    "train": result.train_t1["Appliances"],
    "val":   result.val_t1["Appliances"],
    "test":  result.test_t1["Appliances"],
}

candidate_thresholds = [0, 10, 20, 30, 40, 50, 60]

print("=== Summary statistics (Appliances, raw Wh) ===\n")
for name, series in partitions.items():
    print(f"--- {name} (n={len(series)}) ---")
    print(f"min:    {series.min()}")
    print(f"max:    {series.max()}")
    print(f"mean:   {series.mean():.2f}")
    print(f"median: {series.median()}")
    print(f"std:    {series.std():.2f}")
    print()

print("=== Quantiles ===\n")
quantile_levels = [0.0, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
quantile_table = pd.DataFrame({
    name: series.quantile(quantile_levels) for name, series in partitions.items()
})
quantile_table.index = [f"{int(q*100)}%" for q in quantile_levels]
print(quantile_table)
print()

print("=== Exact zero count ===\n")
for name, series in partitions.items():
    n_zero = (series == 0).sum()
    pct_zero = 100 * n_zero / len(series)
    print(f"{name}: {n_zero} rows == 0 ({pct_zero:.3f}%)")
print()

print("=== Candidate threshold coverage (rows BELOW threshold) ===\n")
threshold_table = pd.DataFrame({
    name: [
        f"{(series < t).sum()} ({100*(series < t).sum()/len(series):.2f}%)"
        for t in candidate_thresholds
    ]
    for name, series in partitions.items()
})
threshold_table.index = [f"< {t}" for t in candidate_thresholds]
print(threshold_table)

=== Summary statistics (Appliances, raw Wh) ===

--- train (n=13866) ---
min:    10
max:    1080
mean:   99.00
median: 60.0
std:    107.06

--- val (n=1728) ---
min:    20
max:    870
mean:   90.90
median: 60.0
std:    89.19

--- test (n=3996) ---
min:    20
max:    850
mean:   95.78
median: 60.0
std:    90.54

=== Quantiles ===

     train    val   test
0%    10.0   20.0   20.0
1%    20.0   30.0   30.0
5%    30.0   40.0   40.0
10%   40.0   40.0   50.0
25%   50.0   50.0   50.0
50%   60.0   60.0   60.0
75%  100.0   90.0  100.0
90%  210.0  130.0  160.0
95%  340.0  280.0  282.5
99%  590.0  524.6  540.0

=== Exact zero count ===

train: 0 rows == 0 (0.000%)
val: 0 rows == 0 (0.000%)
test: 0 rows == 0 (0.000%)

=== Candidate threshold coverage (rows BELOW threshold) ===

              train           val           test
< 0       0 (0.00%)     0 (0.00%)      0 (0.00%)
< 10      0 (0.00%)     0 (0.00%)      0 (0.00%)
< 20      9 (0.06%)     0 (0.00%)      0 (0.00%)
< 30    324 (2.34%)    10 (

In [5]:
import pandas as pd

from config.paths import RAW_DATA_PATH
from config.features import MAPE_THRESHOLD
from src.pipeline import run_pipeline
from src.models.naive import NaivePersistenceForecaster, NaiveSeasonalForecaster
from src.evaluation.harness import evaluate_forecaster

df_raw = pd.read_csv(RAW_DATA_PATH, parse_dates=["date"])
result = run_pipeline(df_raw)

configs = [
    ("naive_persistence", NaivePersistenceForecaster(), 1,
     result.train_t1, result.val_t1, result.test_t1, "target_t1"),
    ("naive_persistence", NaivePersistenceForecaster(), 6,
     result.train_t6, result.val_t6, result.test_t6, "target_t6"),
    ("naive_seasonal", NaiveSeasonalForecaster(horizon=1), 1,
     result.train_t1, result.val_t1, result.test_t1, "target_t1"),
    ("naive_seasonal", NaiveSeasonalForecaster(horizon=6), 6,
     result.train_t6, result.val_t6, result.test_t6, "target_t6"),
]

rows = []
for name, forecaster, horizon, train_df, val_df, test_df, target_col in configs:
    eval_result = evaluate_forecaster(
        forecaster=forecaster,
        train_df=train_df, val_df=val_df, test_df=test_df,
        target_column=target_col,
        model_name=name, horizon=horizon,
        mape_threshold=MAPE_THRESHOLD,
    )
    for partition in ("train", "val", "test"):
        m = eval_result.metrics[partition]
        rows.append({
            "model": name, "horizon": horizon, "partition": partition,
            "mae": round(m["mae"], 2),
            "rmse": round(m["rmse"], 2),
            "mape": round(m["mape"], 2),
        })

leaderboard = pd.DataFrame(rows)
print(leaderboard.to_string(index=False))

            model  horizon partition   mae   rmse  mape
naive_persistence        1     train 30.84  74.17 25.47
naive_persistence        1       val 25.24  65.30 21.78
naive_persistence        1      test 26.50  66.14 21.66
naive_persistence        6     train 60.27 124.12 53.96
naive_persistence        6       val 49.16 107.42 45.54
naive_persistence        6      test 47.60 103.62 40.43
   naive_seasonal        1     train 68.09 134.63 68.75
   naive_seasonal        1       val 48.75 108.04 50.84
   naive_seasonal        1      test 53.72 112.29 51.13
   naive_seasonal        6     train 68.02 134.56 68.73
   naive_seasonal        6       val 48.75 108.04 50.85
   naive_seasonal        6      test 53.78 112.36 51.18


In [6]:
from config.features import MAPE_THRESHOLD
print(MAPE_THRESHOLD)

30.0


In [7]:
import numpy as np
import pandas as pd

from config.features import MAPE_THRESHOLD
from src.models.naive import NaivePersistenceForecaster

partitions = {
    "train": result.train_t1,
    "val":   result.val_t1,
    "test":  result.test_t1,
}

forecaster = NaivePersistenceForecaster()

print(f"MAPE threshold: {MAPE_THRESHOLD}\n")

for name, df in partitions.items():
    X = df[forecaster.required_columns].to_numpy()
    y_true = df["target_t1"].to_numpy()
    y_pred = forecaster.predict(X)

    abs_error = np.abs(y_true - y_pred)

    mape_mask = np.abs(y_true) >= MAPE_THRESHOLD
    y_true_used = y_true[mape_mask]
    y_pred_used = y_pred[mape_mask]
    abs_error_used = abs_error[mape_mask]
    rel_error_used = abs_error_used / np.abs(y_true_used) * 100

    print(f"--- {name} (n={len(df)}) ---")
    print(f"Absolute error (ALL rows): mean={abs_error.mean():.2f}, median={np.median(abs_error):.2f}, std={abs_error.std():.2f}")
    print(f"MAPE-used rows: {mape_mask.sum()} / {len(df)} ({100*mape_mask.sum()/len(df):.2f}%)")
    print(f"  y_true (MAPE-used) distribution: min={y_true_used.min()}, median={np.median(y_true_used):.1f}, mean={y_true_used.mean():.2f}, max={y_true_used.max()}")
    print(f"  y_true (MAPE-used) quantiles: 5%={np.quantile(y_true_used,0.05):.1f}, 25%={np.quantile(y_true_used,0.25):.1f}, 50%={np.quantile(y_true_used,0.5):.1f}, 75%={np.quantile(y_true_used,0.75):.1f}")
    print(f"  abs_error (MAPE-used): mean={abs_error_used.mean():.2f}, median={np.median(abs_error_used):.2f}")
    print(f"  relative_error % (MAPE-used): mean={rel_error_used.mean():.2f}, median={np.median(rel_error_used):.2f}, std={rel_error_used.std():.2f}")
    print()

MAPE threshold: 30.0

--- train (n=13866) ---
Absolute error (ALL rows): mean=30.84, median=10.00, std=67.46
MAPE-used rows: 13542 / 13866 (97.66%)
  y_true (MAPE-used) distribution: min=30.0, median=60.0, mean=100.90, max=1080.0
  y_true (MAPE-used) quantiles: 5%=40.0, 25%=50.0, 50%=60.0, 75%=100.0
  abs_error (MAPE-used): mean=31.30, median=10.00
  relative_error % (MAPE-used): mean=25.47, median=16.67, std=43.50

--- val (n=1728) ---
Absolute error (ALL rows): mean=25.24, median=10.00, std=60.23
MAPE-used rows: 1718 / 1728 (99.42%)
  y_true (MAPE-used) distribution: min=30.0, median=60.0, mean=91.32, max=870.0
  y_true (MAPE-used) quantiles: 5%=40.0, 25%=50.0, 50%=60.0, 75%=90.0
  abs_error (MAPE-used): mean=25.26, median=10.00
  relative_error % (MAPE-used): mean=21.78, median=16.67, std=32.95

--- test (n=3996) ---
Absolute error (ALL rows): mean=26.50, median=10.00, std=60.60
MAPE-used rows: 3981 / 3996 (99.62%)
  y_true (MAPE-used) distribution: min=30.0, median=60.0, mean=96.16

In [8]:
import mlflow

# What tracking URI is actually active RIGHT NOW in a fresh session,
# with no explicit set_tracking_uri() call?
print("Default tracking URI:", mlflow.get_tracking_uri())

Default tracking URI: sqlite:////home/jegan/Desktop/WattCast/notebooks/mlflow.db


/home/jegan/Desktop/WattCast/env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import mlflow

for uri in [
    f"sqlite:///{Path.cwd()}/db/mlflow.db" if (Path.cwd()/"db"/"mlflow.db").exists() else None,
    "sqlite:///./mlflow.db",
    "sqlite:///./notebooks/mlflow.db",
]:
    if uri is None:
        continue
    mlflow.set_tracking_uri(uri)
    client = mlflow.tracking.MlflowClient()
    experiments = client.search_experiments()
    print(f"\n{uri}")
    for exp in experiments:
        runs = client.search_runs(exp.experiment_id)
        print(f"  experiment: {exp.name} ({len(runs)} runs)")

2026/09/23 22:18:09 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/23 22:18:10 INFO mlflow.store.db.utils: Updating database tables
2026/09/23 22:18:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/23 22:18:11 INFO mlflow.store.db.utils: Updating database tables



sqlite:///./mlflow.db
  experiment: Default (0 runs)

sqlite:///./notebooks/mlflow.db
  experiment: Default (0 runs)


In [10]:
import mlflow
from pathlib import Path

PROJECT_ROOT = Path("/home/jegan/Desktop/WattCast")

candidates = {
    "db/mlflow.db (Phase 0 intended)": PROJECT_ROOT / "db" / "mlflow.db",
    "repo root mlflow.db (stray)":     PROJECT_ROOT / "mlflow.db",
    "notebooks/mlflow.db (stray)":     PROJECT_ROOT / "notebooks" / "mlflow.db",
}

for label, path in candidates.items():
    print(f"\n=== {label} ===")
    print(f"Path: {path}")
    print(f"Exists: {path.exists()}")
    if not path.exists():
        continue
    print(f"Size: {path.stat().st_size} bytes")

    uri = f"sqlite:///{path}"
    mlflow.set_tracking_uri(uri)
    client = mlflow.tracking.MlflowClient()
    experiments = client.search_experiments()
    for exp in experiments:
        runs = client.search_runs(exp.experiment_id)
        print(f"  experiment: {exp.name} ({len(runs)} runs)")


=== db/mlflow.db (Phase 0 intended) ===
Path: /home/jegan/Desktop/WattCast/db/mlflow.db
Exists: True
Size: 876544 bytes
  experiment: Default (2 runs)

=== repo root mlflow.db (stray) ===
Path: /home/jegan/Desktop/WattCast/mlflow.db
Exists: True
Size: 876544 bytes
  experiment: Default (10 runs)

=== notebooks/mlflow.db (stray) ===
Path: /home/jegan/Desktop/WattCast/notebooks/mlflow.db
Exists: True
Size: 876544 bytes
  experiment: Default (0 runs)


In [12]:
import mlflow
from pathlib import Path

def inspect_store(label, path):
    print(f"\n{'='*70}")
    print(f"{label}")
    print(f"{'='*70}")

    uri = f"sqlite:///{path}"
    mlflow.set_tracking_uri(uri)
    client = mlflow.tracking.MlflowClient()

    experiments = client.search_experiments()
    for exp in experiments:
        runs = client.search_runs(exp.experiment_id, order_by=["start_time ASC"])
        if not runs:
            continue
        print(f"\n--- Experiment: {exp.name} (id={exp.experiment_id}), {len(runs)} runs ---")

        for run in runs:
            info = run.info
            print(f"\n  Run ID:      {info.run_id}")
            print(f"  Run name:    {run.data.tags.get('mlflow.runName', '(none)')}")
            print(f"  Status:      {info.status}")
            print(f"  Start time:  {pd.to_datetime(info.start_time, unit='ms')}")
            print(f"  End time:    {pd.to_datetime(info.end_time, unit='ms') if info.end_time else '(still running / not closed)'}")
            print(f"  Params:      {dict(run.data.params)}")
            print(f"  Metrics:     {dict(run.data.metrics)}")
            # exclude noisy default mlflow.* tags for readability, show the rest
            user_tags = {k: v for k, v in run.data.tags.items() if not k.startswith("mlflow.")}
            print(f"  Tags:        {user_tags}")
            print(f"  Artifact URI:{info.artifact_uri}")

            try:
                artifacts = client.list_artifacts(info.run_id)
                artifact_paths = [a.path for a in artifacts]
                print(f"  Artifacts:   {artifact_paths}")
            except Exception as e:
                print(f"  Artifacts:   (could not list: {e})")


import pandas as pd

PROJECT_ROOT = Path("/home/jegan/Desktop/WattCast")

inspect_store("REPO ROOT — /mlflow.db (10 runs)", PROJECT_ROOT / "mlflow.db")
inspect_store("db/mlflow.db (2 runs, Phase 0 intended)", PROJECT_ROOT / "db" / "mlflow.db")


REPO ROOT — /mlflow.db (10 runs)

--- Experiment: Default (id=0), 10 runs ---

  Run ID:      90d65813f14141d8b1cc74a3fef24caf
  Run name:    judicious-shrimp-793
  Status:      FINISHED
  Start time:  2026-09-23 14:14:21.143000
  End time:    2026-09-23 14:14:34.558000
  Params:      {}
  Metrics:     {}
  Tags:        {}
  Artifact URI:mlflow-artifacts:/0/90d65813f14141d8b1cc74a3fef24caf/artifacts
  Artifacts:   (could not list: When an mlflow-artifacts URI was supplied, the tracking URI must be a valid http or https URI, but it was currently set to sqlite:////home/jegan/Desktop/WattCast/mlflow.db. Perhaps you forgot to set the tracking URI to the running MLflow server. To set the tracking URI, use either of the following methods:
1. Set the MLFLOW_TRACKING_URI environment variable to the desired tracking URI. `export MLFLOW_TRACKING_URI=http://localhost:5000`
2. Set the tracking URI programmatically by calling `mlflow.set_tracking_uri`. `mlflow.set_tracking_uri('http://localhost:50

In [13]:
import mlflow
from config.mlflow_config import TRACKING_URI, EXPERIMENT_NAME

mlflow.set_tracking_uri(TRACKING_URI)
print("Tracking URI now:", mlflow.get_tracking_uri())

client = mlflow.tracking.MlflowClient()
experiments = client.search_experiments()
for exp in experiments:
    runs = client.search_runs(exp.experiment_id)
    print(f"experiment: {exp.name} ({len(runs)} runs)")

Tracking URI now: sqlite:////home/jegan/Desktop/WattCast/db/mlflow.db
experiment: Default (2 runs)


In [15]:
import tempfile
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression

with tempfile.TemporaryDirectory() as tmp:
    mlflow.set_tracking_uri(f"sqlite:///{tmp}/test.db")
    mlflow.set_experiment("scratch_test")

    model = LinearRegression().fit([[1.0], [2.0]], [1.0, 2.0])

    with mlflow.start_run() as run:
        mlflow.sklearn.log_model(model, name="model")
        run_id = run.info.run_id

    client = mlflow.tracking.MlflowClient()
    artifacts = client.list_artifacts(run_id)
    print([a.path for a in artifacts])

2026/09/23 22:37:31 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/23 22:37:31 INFO mlflow.store.db.utils: Updating database tables
2026/09/23 22:37:32 INFO mlflow.tracking.fluent: Experiment with name 'scratch_test' does not exist. Creating a new experiment.


[]


In [16]:
import tempfile
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression

with tempfile.TemporaryDirectory() as tmp:
    mlflow.set_tracking_uri(f"sqlite:///{tmp}/test.db")
    mlflow.set_experiment("scratch_test")

    model = LinearRegression().fit([[1.0], [2.0]], [1.0, 2.0])

    with mlflow.start_run() as run:
        logged_model_info = mlflow.sklearn.log_model(model, name="model")
        run_id = run.info.run_id

    print("logged_model_info:", logged_model_info)
    print("model_id:", logged_model_info.model_id)
    print("model_uri:", logged_model_info.model_uri)

    # the new MLflow 3.x way to retrieve logged models for a run
    logged_models = mlflow.search_logged_models(
        experiment_ids=[mlflow.get_experiment_by_name("scratch_test").experiment_id],
        filter_string=f"source_run_id='{run_id}'",
    )
    print("\nsearch_logged_models result:")
    print(logged_models)

2026/09/23 22:38:07 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/23 22:38:07 INFO mlflow.store.db.utils: Updating database tables
2026/09/23 22:38:08 INFO mlflow.tracking.fluent: Experiment with name 'scratch_test' does not exist. Creating a new experiment.


logged_model_info: <mlflow.models.model.ModelInfo object at 0x76bcfcb7b610>
model_id: m-c7aa3edd0def441a9ba4daaf54eee775
model_uri: models:/m-c7aa3edd0def441a9ba4daaf54eee775

search_logged_models result:
                                   artifact_location  creation_timestamp  \
0  /home/jegan/Desktop/WattCast/notebooks/mlruns/...       1790183288283   

  experiment_id  last_updated_timestamp metrics  \
0             1           1790183295043    None   

                             model_id model_type   name params  \
0  m-c7aa3edd0def441a9ba4daaf54eee775       None  model     {}   

                      source_run_id status status_message  \
0  fc10f5132dde4be88800765ab6b442cf  READY           None   

                                                tags  
0  {'mlflow.source.name': '03_baselines.ipynb', '...  


In [18]:
import pandas as pd
from pathlib import Path

from config.paths import RAW_DATA_PATH
from config.features import MAPE_THRESHOLD
from src.pipeline import run_pipeline
from src.models.naive import NaivePersistenceForecaster, NaiveSeasonalForecaster
from src.models.sklearn_models import LinearRegressionForecaster, RandomForestForecaster
from src.evaluation.harness import evaluate_forecaster
from src.tracking.mlflow_logger import log_evaluation_result

# Reuse the already-generated Phase 2 pipeline output — no regeneration
df_raw = pd.read_csv(RAW_DATA_PATH, parse_dates=["date"])
result = run_pipeline(df_raw)

SCALER_PATH = str(Path.cwd().parent / "data" / "processed" / "scaler_train_fit.joblib")

configs = [
    ("naive_persistence", NaivePersistenceForecaster(), 1,
     result.train_t1, result.val_t1, result.test_t1, "target_t1"),
    ("naive_persistence", NaivePersistenceForecaster(), 6,
     result.train_t6, result.val_t6, result.test_t6, "target_t6"),
    ("naive_seasonal", NaiveSeasonalForecaster(horizon=1), 1,
     result.train_t1, result.val_t1, result.test_t1, "target_t1"),
    ("naive_seasonal", NaiveSeasonalForecaster(horizon=6), 6,
     result.train_t6, result.val_t6, result.test_t6, "target_t6"),
    ("linear_regression", LinearRegressionForecaster(), 1,
     result.train_t1, result.val_t1, result.test_t1, "target_t1"),
    ("linear_regression", LinearRegressionForecaster(), 6,
     result.train_t6, result.val_t6, result.test_t6, "target_t6"),
    ("random_forest", RandomForestForecaster(), 1,
     result.train_t1, result.val_t1, result.test_t1, "target_t1"),
    ("random_forest", RandomForestForecaster(), 6,
     result.train_t6, result.val_t6, result.test_t6, "target_t6"),
]

rows = []
run_ids = {}

for name, forecaster, horizon, train_df, val_df, test_df, target_col in configs:
    eval_result = evaluate_forecaster(
        forecaster=forecaster,
        train_df=train_df, val_df=val_df, test_df=test_df,
        target_column=target_col,
        model_name=name, horizon=horizon,
        mape_threshold=MAPE_THRESHOLD,
    )

    # native_model only exists for the sklearn-backed wrappers
    native_model = getattr(forecaster, "model", None)

    run_id = log_evaluation_result(
        result=eval_result,
        forecaster=forecaster,
        mape_threshold=MAPE_THRESHOLD,
        scaler_path=SCALER_PATH,
        native_model=native_model,
    )
    run_ids[f"{name}_h{horizon}"] = run_id

    for partition in ("train", "val", "test"):
        m = eval_result.metrics[partition]
        rows.append({
            "model": name, "horizon": horizon, "partition": partition,
            "mae": round(m["mae"], 2),
            "rmse": round(m["rmse"], 2),
            "mape": round(m["mape"], 2),
        })

leaderboard = pd.DataFrame(rows)
print(leaderboard.to_string(index=False))
print("\nMLflow run IDs:")
for key, rid in run_ids.items():
    print(f"  {key}: {rid}")

2026/09/23 22:43:55 INFO mlflow.tracking.fluent: Experiment with name 'WattCast' does not exist. Creating a new experiment.


MlflowException: The saved sklearn model references untrusted types. If you are sure loading these types is safe, set the 'skops_trusted_types' parameter when calling 'log_model' or 'save_model' to the list of trusted types. Root error: Untrusted types found in the file: ['sklearn.tree._tree.Tree'].

- sklearn.tree._tree.Tree: sklearn.tree._tree.Tree (the shared node storage for DecisionTree*, RandomForest*, ExtraTrees*, and GradientBoosting* models) stores raw node indices (left_child, right_child, feature) that scikit-learn indexes into without bounds checking. A malicious file can set these to out-of-range values: skops loads the object successfully, but calling .predict() on it can then crash the process (segfault) or read out-of-bounds memory. If you created the file yourself or otherwise fully trust its source, you can load it with trusted=["sklearn.tree._tree.Tree"].

Only add the specific types you have reviewed and trust to the `trusted` argument; avoid passing everything repo

In [19]:
# quick verification: does the fixed logger handle a real RandomForestForecaster?
import importlib
import src.tracking.mlflow_logger
importlib.reload(src.tracking.mlflow_logger)  # picks up the file edit without a kernel restart... verify this actually works
from src.tracking.mlflow_logger import log_evaluation_result

In [20]:
import pandas as pd
from pathlib import Path

from config.paths import RAW_DATA_PATH
from config.features import MAPE_THRESHOLD
from src.pipeline import run_pipeline
from src.models.naive import NaivePersistenceForecaster, NaiveSeasonalForecaster
from src.models.sklearn_models import LinearRegressionForecaster, RandomForestForecaster
from src.evaluation.harness import evaluate_forecaster
from src.tracking.mlflow_logger import log_evaluation_result

df_raw = pd.read_csv(RAW_DATA_PATH, parse_dates=["date"])
result = run_pipeline(df_raw)

SCALER_PATH = str(Path.cwd().parent / "data" / "processed" / "scaler_train_fit.joblib")

configs = [
    ("naive_persistence", NaivePersistenceForecaster(), 1,
     result.train_t1, result.val_t1, result.test_t1, "target_t1"),
    ("naive_persistence", NaivePersistenceForecaster(), 6,
     result.train_t6, result.val_t6, result.test_t6, "target_t6"),
    ("naive_seasonal", NaiveSeasonalForecaster(horizon=1), 1,
     result.train_t1, result.val_t1, result.test_t1, "target_t1"),
    ("naive_seasonal", NaiveSeasonalForecaster(horizon=6), 6,
     result.train_t6, result.val_t6, result.test_t6, "target_t6"),
    ("linear_regression", LinearRegressionForecaster(), 1,
     result.train_t1, result.val_t1, result.test_t1, "target_t1"),
    ("linear_regression", LinearRegressionForecaster(), 6,
     result.train_t6, result.val_t6, result.test_t6, "target_t6"),
    ("random_forest", RandomForestForecaster(), 1,
     result.train_t1, result.val_t1, result.test_t1, "target_t1"),
    ("random_forest", RandomForestForecaster(), 6,
     result.train_t6, result.val_t6, result.test_t6, "target_t6"),
]

rows = []
run_ids = {}

for name, forecaster, horizon, train_df, val_df, test_df, target_col in configs:
    eval_result = evaluate_forecaster(
        forecaster=forecaster,
        train_df=train_df, val_df=val_df, test_df=test_df,
        target_column=target_col,
        model_name=name, horizon=horizon,
        mape_threshold=MAPE_THRESHOLD,
    )

    native_model = getattr(forecaster, "model", None)

    run_id = log_evaluation_result(
        result=eval_result,
        forecaster=forecaster,
        mape_threshold=MAPE_THRESHOLD,
        scaler_path=SCALER_PATH,
        native_model=native_model,
    )
    run_ids[f"{name}_h{horizon}"] = run_id

    for partition in ("train", "val", "test"):
        m = eval_result.metrics[partition]
        rows.append({
            "model": name, "horizon": horizon, "partition": partition,
            "mae": round(m["mae"], 2),
            "rmse": round(m["rmse"], 2),
            "mape": round(m["mape"], 2),
        })

leaderboard = pd.DataFrame(rows)
print(leaderboard.to_string(index=False))
print("\nMLflow run IDs:")
for key, rid in run_ids.items():
    print(f"  {key}: {rid}")

            model  horizon partition   mae   rmse  mape
naive_persistence        1     train 30.84  74.17 25.47
naive_persistence        1       val 25.24  65.30 21.78
naive_persistence        1      test 26.50  66.14 21.66
naive_persistence        6     train 60.27 124.12 53.96
naive_persistence        6       val 49.16 107.42 45.54
naive_persistence        6      test 47.60 103.62 40.43
   naive_seasonal        1     train 68.09 134.63 68.75
   naive_seasonal        1       val 48.75 108.04 50.84
   naive_seasonal        1      test 53.72 112.29 51.13
   naive_seasonal        6     train 68.02 134.56 68.73
   naive_seasonal        6       val 48.75 108.04 50.85
   naive_seasonal        6      test 53.78 112.36 51.18
linear_regression        1     train 31.54  67.70 29.93
linear_regression        1       val 26.16  58.75 26.64
linear_regression        1      test 27.16  59.71 24.90
linear_regression        6     train 51.10  94.21 52.89
linear_regression        6       val 44.41  81.9

In [21]:
import mlflow
from config.mlflow_config import TRACKING_URI, EXPERIMENT_NAME

mlflow.set_tracking_uri(TRACKING_URI)
client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name(EXPERIMENT_NAME)
runs = client.search_runs(exp.experiment_id, order_by=["start_time ASC"])

print(f"Total runs in '{EXPERIMENT_NAME}' experiment: {len(runs)}\n")

for run in runs:
    run_name = run.data.tags.get("mlflow.runName", "?")
    n_metrics = len(run.data.metrics)
    has_git_sha = bool(run.data.tags.get("git_commit_sha"))
    artifacts = [a.path for a in client.list_artifacts(run.info.run_id)]

    logged_models = mlflow.search_logged_models(
        experiment_ids=[exp.experiment_id],
        filter_string=f"source_run_id='{run.info.run_id}'",
    )
    has_model = len(logged_models) > 0

    print(f"{run_name}: metrics={n_metrics}/9, git_sha={'✓' if has_git_sha else '✗'}, "
          f"artifacts={artifacts}, native_model_logged={'✓' if has_model else '✗ (expected for naive baselines)'}")

Total runs in 'WattCast' experiment: 15

naive_persistence_h1: metrics=9/9, git_sha=✓, artifacts=['predictions.json', 'scaler'], native_model_logged=✗ (expected for naive baselines)
naive_persistence_h6: metrics=9/9, git_sha=✓, artifacts=['predictions.json', 'scaler'], native_model_logged=✗ (expected for naive baselines)
naive_seasonal_h1: metrics=9/9, git_sha=✓, artifacts=['predictions.json', 'scaler'], native_model_logged=✗ (expected for naive baselines)
naive_seasonal_h6: metrics=9/9, git_sha=✓, artifacts=['predictions.json', 'scaler'], native_model_logged=✗ (expected for naive baselines)
linear_regression_h1: metrics=9/9, git_sha=✓, artifacts=['predictions.json', 'scaler'], native_model_logged=✓
linear_regression_h6: metrics=9/9, git_sha=✓, artifacts=['predictions.json', 'scaler'], native_model_logged=✓
random_forest_h1: metrics=9/9, git_sha=✓, artifacts=['predictions.json', 'scaler'], native_model_logged=✓
naive_persistence_h1: metrics=9/9, git_sha=✓, artifacts=['predictions.json'